# Procesador de Variables Buró de Crédito
## Modelo Multimoney - Formato XML GMR_DATALIST

## Descripción General

Este script procesa archivos JSON provenientes de consultas de Buró de Crédito y extrae exclusivamente las variables definidas como **"Entrada 0"** para el modelo **Multimoney**, generando un archivo XML en formato **GMR_DATALIST** listo para scoring.

---

## Objetivo

Generar un **XML estructurado en formato GMR_DATALIST** que contiene:
- Variables agregadas del Buró de Crédito
- Identificadores de trazabilidad (IDCLIENTE e IDSOLICITUD)
- Solo variables con información válida (sin valores NaN o vacíos)

---

## Variables Generadas

### Identificadores
- **IDCLIENTE**: `numeroControlConsulta` del Buró de Crédito
- **IDSOLICITUD**: ID personalizado opcional para trazabilidad (se usa en GMR_IDELEMENTO)

### Variables Financieras
- **CREDMAXAUTCONS**: Crédito máximo autorizado en consumo
- **MONTOPAGARCONS**: Monto a pagar en consumo
- **SALDOACTCONS**: Saldo actual en consumo
- **CREDMAXAUTREV**: Crédito máximo autorizado revolvente
- **SALDOACTREV**: Saldo actual revolvente
- **LIMITECREDITO**: Límite de crédito
- **SALDOACTTDC**: Saldo actual en tarjetas de crédito
- **CREDMAXAUTTOTAL**: Crédito máximo autorizado total
- **SALDOVENCTOTAL**: Saldo vencido total

### Variables Temporales
- **FECHAREPORTE**: Fecha de solicitud más reciente (formato YYYYMMDD)
- **FECHAULTCONSULTA**: Fecha de última consulta (formato YYYYMMDD)

### Variables de Comportamiento
- **PEORMOP**: Peor MOP (Manner of Payment) histórico
- **PEORHIST01** a **PEORHIST24**: Peor comportamiento de pago por mes (últimos 24 meses)

---

## Metodología de Cálculo

### 1. Limpieza de Datos
Los montos se normalizan eliminando:
- Símbolos `"+"`
- Cadenas vacías o valores nulos
- Conversión a tipos numéricos válidos

### 2. Agregación por Tipo de Contrato
Los agregados se calculan según **`tipoContrato`**, conforme al **Anexo 3 del Manual API Reporte de Crédito**:
- **CONSUMO**: AN, AG, AL, AP, AU, BD, BT, CE, CF, CO, CS, CT, DE, EQ, FI, FT, HA, HE, HI, LS, MI, OA, PA, PB, PG, PL, PN, PQ, PR, PS, RC, RD, RE, RF, RN, RV, SE, SG, SM, ST, UK, US
  - Variables: CREDMAXAUTCONS, MONTOPAGARCONS, SALDOACTCONS
- **REVOLVENTE**: CL, LR
  - Variables: CREDMAXAUTREV, SALDOACTREV
- **TDC (Tarjetas de Crédito)**: CC, SC, TE
  - Variables: SALDOACTTDC, LIMITECREDITO

### 3. Cálculo de PEORMOP
**PEORMOP** se obtiene como el **máximo** entre:
- `formaPagoActual`
- `mopHistoricoMorosidadMasGrave`

### 4. Cálculo de PEORHIST01 a PEORHIST24
- Se toman los **últimos 24 caracteres** de `historicoPagos`
- Se **alinean desde el mes más reciente** (PEORHIST01 = mes actual)
- Se calcula el **peor valor por posición** entre todas las cuentas reportadas

### 5. Filtrado de Variables
- Solo se incluyen variables con valores válidos (no NaN, no vacíos)
- Se generan advertencias para cada variable excluida
- IDCLIENTE siempre se incluye

---

## Formato de Salida XML

```xml
<GMR_DATALIST>
  <GMR_DATA>
    <GMR_HEADER>
      <GMR_FLOW>1</GMR_FLOW>
      <GMR_VERSION>0</GMR_VERSION>
      <GMR_RESERVED>00</GMR_RESERVED>
      <GMR_LEVELS>1</GMR_LEVELS>
      <GMR_IDELEMENTO>[IDSOLICITUD o IDCLIENTE]</GMR_IDELEMENTO>
    </GMR_HEADER>
    <ClienteIn>
      <IDCLIENTE>[numeroControlConsulta]</IDCLIENTE>
      <!-- Variables del modelo -->
    </ClienteIn>
  </GMR_DATA>
</GMR_DATALIST>
```

---

## Sistema de Advertencias

El sistema genera advertencias cuando:
1. No se encuentra `numeroControlConsulta` en el JSON
2. Una variable no tiene información disponible
3. El JSON no contiene cuentas reportadas
4. El JSON no contiene historial de consultas

---

## Uso

```python
# Sin IDSOLICITUD (usa IDCLIENTE en GMR_IDELEMENTO)
resultado, warnings = procesar_json("ruta/archivo.json")

# Con IDSOLICITUD personalizado
resultado, warnings = procesar_json("ruta/archivo.json", "SOL-2026-001")

# Generar XML
xml_output = generar_xml(resultado)
```

---

## Pipeline de Procesamiento

Este script forma parte del **pipeline previo al modelo predictivo de probabilidad de mora (90+ días)**:

1. **Entrada**: JSON del Buró de Crédito
2. **Procesamiento**: Extracción y agregación de variables
3. **Validación**: Filtrado de variables sin información
4. **Salida**: XML en formato GMR_DATALIST
5. **Scoring**: Consumo por el modelo predictivo

El resultado queda **listo para ser consumido por el modelo de scoring**.

In [1]:
# Librerías
import json
import re
import numpy as np
from datetime import datetime
import xml.etree.ElementTree as ET
from xml.dom import minidom

In [2]:
# Configuración Anexo 3
CONSUMO = ["AN", "AG", "AL", "AP", "AU", "BD", "BT", "CE", "CF", "CO", "CS", "CT", "DE", "EQ"
           , "FI", "FT", "HA", "HE", "HI", "LS", "MI", "OA", "PA", "PB", "PG", "PL","PN", "PQ",
           "PR", "PS", "RC", "RD", "RE", "RF", "RN", "RV", "SE", "SG", "SM", "ST", "UK", "US"]
REVOLVENTE = ["CL", "LR"]
TDC = ["CC", "SC", "TE"]


In [3]:
def limpiar_numero(valor):

    if valor is None or valor == "":
        return np.nan

    valor = str(valor).replace("+", "").strip()

    try:
        return int(valor)
    except:
        return np.nan


def normalizar_fecha(fecha):

    """
    Convierte fechas a formato YYYYMMDD.
    Si no puede interpretarse, devuelve NaN.
    """

    if fecha is None or fecha == "":
        return np.nan

    fecha = str(fecha)

    formatos = [
        "%d%m%Y",
        "%Y%m%d",
        "%Y-%m-%d",
        "%d-%m-%Y"
    ]

    for f in formatos:
        try:
            return datetime.strptime(fecha, f).strftime("%Y%m%d")
        except:
            pass

    return np.nan


def obtener_id_consulta(json_data, warnings):

    try:
        id_consulta = json_data["respuesta"]["persona"]["encabezado"].get("numeroControlConsulta")
        
        if id_consulta is None or id_consulta == "":
            warnings.append("No se encontró 'numeroControlConsulta' en el JSON. Se usará 'id_no_disponible' como identificador")
            return "id_no_disponible"
        
        return id_consulta

    except:
        warnings.append("Error al acceder a 'numeroControlConsulta' en el JSON. Se usará 'id_no_disponible' como identificador")
        return "id_no_disponible"


def cargar_json_seguro(path_json):

    with open(path_json, "r", encoding="utf-8") as f:
        contenido = f.read()

    try:
        return json.loads(contenido)

    except json.JSONDecodeError:

        contenido = re.sub(
            r'("valorScore"\s*:\s*"\d+")\s*("codigoRazon")',
            r'\1,\2',
            contenido
        )

        return json.loads(contenido)


def peor_mop(cuentas):

    peor = np.nan

    for c in cuentas:

        for campo in ["formaPagoActual", "mopHistoricoMorosidadMasGrave"]:

            mop = c.get(campo)

            if mop and str(mop).isdigit():

                mop = int(mop)

                if np.isnan(peor) or mop > peor:
                    peor = mop

    return peor


def fecha_ult_consulta(consultas):

    fechas = []

    for c in consultas:

        fecha = c.get("fechaConsulta")

        if fecha:
            try:
                fechas.append(datetime.strptime(str(fecha), "%d%m%Y"))
            except:
                try:
                    fechas.append(datetime.strptime(str(fecha), "%Y%m%d"))
                except:
                    pass

    if fechas:
        return max(fechas).strftime("%Y%m%d")

    return np.nan


def calcular_peor_historico_1_24(cuentas):

    peor_por_mes = [np.nan] * 24

    for c in cuentas:

        hist = str(c.get("historicoPagos", ""))

        if hist == "":
            continue

        ultimos_24 = hist[-24:].rjust(24, "0")

        for i in range(24):

            ch = ultimos_24[-(i + 1)]

            if ch.isdigit():

                ch = int(ch)

                if np.isnan(peor_por_mes[i]) or ch > peor_por_mes[i]:
                    peor_por_mes[i] = ch

    resultado = {}

    for i in range(24):
        resultado[f"PEOR_HIST_{i+1}"] = peor_por_mes[i]

    return resultado


def procesar_json(path_json, id_solicitud_externo=None):

    warnings = []

    data = cargar_json_seguro(path_json)

    try:
        persona = data["respuesta"]["persona"]
    except:
        raise ValueError("El JSON no contiene la estructura esperada del Buró")

    cuentas = persona.get("cuentas", [])
    consultas = persona.get("consultaEfectuadas", [])
    resumen = persona.get("resumenReporte", [{}])[0]

    if not cuentas:
        warnings.append("El JSON no contiene cuentas reportadas")

    if not consultas:
        warnings.append("El JSON no contiene historial de consultas")

    id_cliente = obtener_id_consulta(data, warnings)

    CREDMAXAUTCONS = np.nan
    MONTOPAGARCONS = np.nan
    SALDOACTCONS = np.nan
    CREDMAXAUTREV = np.nan
    SALDOACTREV = np.nan
    LIMITECREDITO = np.nan
    SALDOACTTDC = np.nan
    CREDMAXAUTTOTAL = np.nan
    SALDOVENCTOTAL = np.nan

    for c in cuentas:

        tipo = c.get("tipoContrato")

        creditoMaximo = limpiar_numero(c.get("creditoMaximo"))
        saldoActual = limpiar_numero(c.get("saldoActual"))
        saldoVencido = limpiar_numero(c.get("saldoVencido"))
        limiteCredito = limpiar_numero(c.get("limiteCredito"))
        montoPagar = limpiar_numero(c.get("montoPagar"))

        if not np.isnan(creditoMaximo):
            CREDMAXAUTTOTAL = np.nansum([CREDMAXAUTTOTAL, creditoMaximo])

        if not np.isnan(saldoVencido):
            SALDOVENCTOTAL = np.nansum([SALDOVENCTOTAL, saldoVencido])

        if tipo in CONSUMO:

            CREDMAXAUTCONS = np.nansum([CREDMAXAUTCONS, creditoMaximo])
            MONTOPAGARCONS = np.nansum([MONTOPAGARCONS, montoPagar])
            SALDOACTCONS = np.nansum([SALDOACTCONS, saldoActual])

        if tipo in REVOLVENTE:

            CREDMAXAUTREV = np.nansum([CREDMAXAUTREV, creditoMaximo])
            SALDOACTREV = np.nansum([SALDOACTREV, saldoActual])

        if tipo in TDC:

            SALDOACTTDC = np.nansum([SALDOACTTDC, saldoActual])
            LIMITECREDITO = np.nansum([LIMITECREDITO, limiteCredito])

    # Preparar todas las variables con sus valores
    variables_completas = {
        "IDCLIENTE": id_cliente,
        "CREDMAXAUTCONS": CREDMAXAUTCONS,
        "MONTOPAGARCONS": MONTOPAGARCONS,
        "SALDOACTCONS": SALDOACTCONS,
        "CREDMAXAUTREV": CREDMAXAUTREV,
        "SALDOACTREV": SALDOACTREV,
        "LIMITECREDITO": LIMITECREDITO,
        "SALDOACTTDC": SALDOACTTDC,
        "CREDMAXAUTTOTAL": CREDMAXAUTTOTAL,
        "SALDOVENCTOTAL": SALDOVENCTOTAL,
        "FECHAREPORTE": normalizar_fecha(resumen.get("fechaSolicitudReporteMasReciente")),
        "FECHAULTCONSULTA": fecha_ult_consulta(consultas),
        "PEORMOP": peor_mop(cuentas)
    }

    # Agregar PEORHIST01 a PEORHIST24
    peor_hist = calcular_peor_historico_1_24(cuentas)
    for i in range(1, 25):
        old_key = f"PEOR_HIST_{i}"
        new_key = f"PEORHIST{i:02d}"
        variables_completas[new_key] = peor_hist.get(old_key, np.nan)

    # Filtrar variables: solo incluir las que tienen información válida
    resultado = {}
    
    for variable, valor in variables_completas.items():
        # Siempre incluir IDCLIENTE
        if variable == "IDCLIENTE":
            resultado[variable] = valor
        # Para el resto, verificar si tiene información válida
        elif isinstance(valor, (int, float)) and not np.isnan(valor):
            resultado[variable] = valor
        elif isinstance(valor, str) and valor != "":
            resultado[variable] = valor
        else:
            # Generar advertencia para variables sin información
            warnings.append(f"Variable '{variable}' no tiene información disponible y no será incluida en la salida")

    # Agregar IDSOLICITUD si fue proporcionado
    if id_solicitud_externo:
        resultado["IDSOLICITUD"] = id_solicitud_externo

    return resultado, warnings


def generar_xml(datos):
    """
    Genera XML en el formato GMR_DATALIST requerido
    """
    # Crear estructura raíz
    root = ET.Element("GMR_DATALIST")
    gmr_data = ET.SubElement(root, "GMR_DATA")
    
    # Header
    gmr_header = ET.SubElement(gmr_data, "GMR_HEADER")
    ET.SubElement(gmr_header, "GMR_FLOW").text = "1"
    ET.SubElement(gmr_header, "GMR_VERSION").text = "0"
    ET.SubElement(gmr_header, "GMR_RESERVED").text = "00"
    ET.SubElement(gmr_header, "GMR_LEVELS").text = "1"
    
    # GMR_IDELEMENTO: usar IDSOLICITUD si está disponible, sino usar IDCLIENTE
    gmr_idelemento = datos.get("IDSOLICITUD", datos.get("IDCLIENTE", "0"))
    ET.SubElement(gmr_header, "GMR_IDELEMENTO").text = str(gmr_idelemento)
    
    # ClienteIn con los datos
    cliente_in = ET.SubElement(gmr_data, "ClienteIn")
    
    # Orden específico de las variables según el formato requerido
    orden_variables = [
        "IDCLIENTE", "CREDMAXAUTCONS", "MONTOPAGARCONS", "SALDOACTCONS",
        "CREDMAXAUTREV", "SALDOACTREV", "LIMITECREDITO", "SALDOACTTDC",
        "CREDMAXAUTTOTAL", "SALDOVENCTOTAL", "FECHAREPORTE", "FECHAULTCONSULTA",
        "PEORMOP"
    ]
    
    # Agregar PEORHIST01 a PEORHIST24
    for i in range(1, 25):
        orden_variables.append(f"PEORHIST{i:02d}")
    
    # Agregar elementos en orden (IDSOLICITUD no se incluye en ClienteIn, solo en GMR_IDELEMENTO)
    for variable in orden_variables:
        if variable in datos:
            valor = datos[variable]
            # Formatear valores numéricos
            if isinstance(valor, float):
                # Si es entero, mostrar sin decimales
                if valor.is_integer():
                    valor_str = str(int(valor))
                else:
                    valor_str = f"{valor:.2f}"
            else:
                valor_str = str(valor)
            
            ET.SubElement(cliente_in, variable).text = valor_str
    
    # Convertir a string con formato bonito
    xml_str = minidom.parseString(ET.tostring(root, encoding='unicode')).toprettyxml(indent="  ")
    
    # Eliminar la declaración XML si existe y líneas vacías
    lines = [line for line in xml_str.split('\n') if line.strip()]
    if lines and lines[0].startswith('<?xml'):
        lines = lines[1:]
    
    return '\n'.join(lines)

In [4]:
# Ejecución
if __name__ == "__main__":

    input_path = "json/RCO_Int_007_Hawk_Response.json"
    output_path = "output_modelo_multimoney.xml"
    
    # Opcional: definir IDSOLICITUD personalizado
    id_solicitud_externo = None  # Cambiar a un valor como "SOL-2026-001" si se desea

    resultado, warnings = procesar_json(input_path, id_solicitud_externo)
    
    # Mostrar advertencias si existen
    if warnings:
        print("\n=== ADVERTENCIAS ===")
        for w in warnings:
            print(f"⚠ {w}")
        print()
    
    # Generar XML
    xml_output = generar_xml(resultado)

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(xml_output)
    
    if id_solicitud_externo:
        print(f"✓ IDSOLICITUD (GMR_IDELEMENTO): {id_solicitud_externo}")
    else:
        print(f"✓ GMR_IDELEMENTO: {resultado.get('IDCLIENTE', 'N/A')}")
    
    print(f"✓ Archivo guardado en: {output_path}")
    print("\n=== XML GENERADO ===")
    print(xml_output)

[
    {
        "id_solicitud": "8c74ac0ef524bfe9",
        "CREDMAXAUTCONS": 1060020.0,
        "MONTOPAGARCONS": 23730.0,
        "SALDOACTCONS": 596709.0,
        "CREDMAXAUTREV": 24758.0,
        "SALDOACTREV": 548.0,
        "LIMITECREDITO": 41656.0,
        "SALDOACTTDC": 55587.0,
        "CREDMAXAUTTOTAL": 1324645.0,
        "SALDOVENCTOTAL": 112469.0,
        "FECHA": "20260127",
        "FECHAULTCONSULTA": "20260127",
        "PEORMOP": 96,
        "PEOR_HIST_1": 6,
        "PEOR_HIST_2": 7,
        "PEOR_HIST_3": 7,
        "PEOR_HIST_4": 7,
        "PEOR_HIST_5": 7,
        "PEOR_HIST_6": 7,
        "PEOR_HIST_7": 7,
        "PEOR_HIST_8": 7,
        "PEOR_HIST_9": 7,
        "PEOR_HIST_10": 7,
        "PEOR_HIST_11": 9,
        "PEOR_HIST_12": 9,
        "PEOR_HIST_13": 9,
        "PEOR_HIST_14": 7,
        "PEOR_HIST_15": 7,
        "PEOR_HIST_16": 7,
        "PEOR_HIST_17": 7,
        "PEOR_HIST_18": 7,
        "PEOR_HIST_19": 7,
        "PEOR_HIST_20": 7,
        "PEOR_H